# OE-pHNN paper comparisons

Select one of the six paper experiments. The runner supplies the complete PH-ablation-derived configuration; this notebook changes only the explicitly selected overrides. Baseline horizons: **MSD 200**, **CT 60**. Best CT horizons (not baseline settings): **AG 60**, **MGM 120**.

The data files must be present at the paths configured in `oeph_runner.py`. Full runs train 10 paired seeds and can take substantial time.

In [ ]:
# ==================== 1. EXPERIMENT SELECTION ====================
# Choose exactly ONE of the six paper experiments:
#   "msd_baseline"             : MSD baseline, prediction horizon 200
#   "ct_baseline"              : CT baseline, prediction horizon 60
#   "msd_snr"                  : MSD training-noise SNR sweep
#   "msd_training_data_size"   : MSD training-data-size sweep
#   "msd_prediction_horizon"  : MSD prediction-horizon sweep
#   "ct_prediction_horizon"   : CT prediction-horizon sweep
PRESET = "msd_baseline"

# ==================== 2. OPTIONAL STUDY OVERRIDES ====================
# Defaults below reproduce the runner's paper configurations.
# Edit a value to override it; use None / {} where indicated to inherit defaults.

# Number of paired model seeds, using the FIRST N seeds from the archived
# 10-seed list. Valid range: 1 through 10. Example: 3 for a short test.
NUM_SEEDS = 10

# Which Hamiltonian methods to train and compare. Use either method alone,
# or both. The paper notebook supports ONLY these two methods.
# Examples: ("autograd",)  or  ("mgradnet_m",)
METHODS = ("autograd", "mgradnet_m")

# Sweep values for the selected ablation; None = original paper levels.
#   PRESET = "msd_snr":                 (15.0, 35.0, 40.0, 50.0)  [dB]
#   PRESET = "msd_training_data_size":  (1, 2, 5, 10, 20)       [1000-sample segments]
#   PRESET = "msd_prediction_horizon": (20, 50, 100, 200)      [samples]
#   PRESET = "ct_prediction_horizon":  (15, 30, 60, 120)       [samples]
# Examples: LEVELS = (15.0, 35.0) for an MSD SNR comparison;
#           LEVELS = (1, 5, 20) for MSD training-data sizes;
#           LEVELS = (60, 120) for CT prediction horizons.
# IMPORTANT: LEVELS has NO effect on baseline presets; baseline horizons
# remain MSD=200 and CT=60. Use None for the complete paper sweep.
LEVELS = None

# Results directory name (under the runner's RESULTS_ROOT).
# None -> "paper_<preset>"; e.g. "paper_msd_baseline".
# Example: "msd_snr_three_seeds" to avoid mixing different runs.
EXPERIMENT_TITLE = None

# ==================== 3. OPTIONAL OPTIMIZER OVERRIDES ====================
# Dictionary keyed by METHOD, then by OptimizerConfig field.
# Overrides affect ONLY the named method; other methods keep their defaults.
# {} = use the original method-specific optimizer settings.
# Valid fields include: name, learning_rate, learning_rate_schedule,
# weight_decay, betas, eps, amsgrad, scheduler_type, scheduler_parameters.
# Examples (uncomment/replace the {} below):
#   OPTIMIZER_OVERRIDES = {"autograd": {"learning_rate": 0.0005}}
#   OPTIMIZER_OVERRIDES = {"mgradnet_m": {
#       "learning_rate": 0.01,
#       "learning_rate_schedule": {0: 0.01, 20: 0.005, 40: 0.001},
#   }}
#   OPTIMIZER_OVERRIDES = {"autograd": {"learning_rate_schedule": None}}
# A schedule maps ZERO-BASED epoch indices to learning rates.
# A method must also appear in METHODS to receive an override.
OPTIMIZER_OVERRIDES = {}

# ==================== 4. OPTIONAL TRAINING OVERRIDES ====================
# Dictionary of TrainingConfig fields. Applies to ALL selected methods.
# {} = use the paper training defaults (MSD: 70 epochs; CT: 700 epochs).
# Valid fields include: epochs, batch_size, early_stopping_patience,
# validation_mode, validation_full_rollout_weight, fit_phases,
# prediction_horizon_schedule, gradient_clip_norm, drop_last,
# validate_initial, restore_history_between_phases.
# Examples (uncomment/replace the {} below):
#   TRAINING_OVERRIDES = {"batch_size": 64}
#   TRAINING_OVERRIDES = {"early_stopping_patience": 20}
#   TRAINING_OVERRIDES = {"validation_mode": "full_rollout"}
#   TRAINING_OVERRIDES = {
#       "validation_mode": "mixed",
#       "validation_full_rollout_weight": 0.1,
#   }
#   TRAINING_OVERRIDES = {"fit_phases": (10, 20, 40)}
# CAUTION: changing epochs, fit_phases, or prediction_horizon_schedule
# changes the training protocol. Keep related settings consistent.
# For PH sweeps, the runner uses a fixed horizon at each LEVELS value.
TRAINING_OVERRIDES = {}


In [ ]:
import oeph_runner as runner

config = runner.paper_configuration(
    PRESET,
    num_seeds=NUM_SEEDS,
    methods=METHODS,
    levels=LEVELS,
    optimizers=OPTIMIZER_OVERRIDES,
    training_overrides=TRAINING_OVERRIDES,
    title=EXPERIMENT_TITLE,
)
print("System:", config.data.system)
print("Study:", config.study.study_type)
print("Methods:", config.study.methods)
print("Seeds:", config.study.explicit_model_seeds)
print("Baseline horizon:", config.windows.prediction_horizon)
print("History:", config.windows.history_length)
print("Validation:", config.training.validation_mode)
print("Output:", config.study.results_root / config.study.study_name)


In [ ]:
# Run the selected study (training and evaluation may take considerable time).
from oeph_training import execute_study
execute_study(config)
